In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd
import random
import json
import time
import re

In [2]:
gsm8k = load_dataset("openai/gsm8k", "main", split="test")
arc = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
gsm_train = load_dataset("openai/gsm8k", "main", split="train")
arc_train = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="train")
K = 3

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

ARC-Challenge/validation-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B / 55.7kB            

ARC-Challenge/validation-00000-of-00001.(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

In [3]:
random.seed(42)
gsm_idx = random.sample(range(len(gsm8k)), 100)
arc_idx = random.sample(range(len(arc)), 100)
print(len(gsm8k), len(arc))

1319 1172


In [4]:
def generate(model,tokenizer, user_prompt, max_new_tokens):
    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens,do_sample=False, temperature=None, top_p=None, top_k=None)
    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True), len(new_tokens)

In [5]:
def gsm_prompt(q, mode):
    if mode == "direct":
        return f"{q}\nReply with only the final number, nothing else. Provide only the answer."
    if mode == "fewshot":
        return f"Solve the problems step by step. At the end write 'Answer: <number>'.\n\nExamples:{gsm_shots}\n\nQuestion: {q}"
    return f"{q}\nSolve step by step. At the end write 'Answer: <number>'."
    
def arc_prompt(q, options, mode):
    opts = "\n".join(f"{chr(65+i)}. {o}" for i, o in enumerate(options))
    base = f"Question: {q}\n{opts}\n"
    if mode == "direct":
        return "Reply with only the letter of the correct option. Provide only the answer." + base
    if mode == "fewshot":
        return f"Think step by step. At the end write 'Answer: <letter>'.\n\n{arc_shots}\n\nQuestion: {q}\n{opts}\nReasoning:"
    return "Think step by step. At the end write 'Answer: <letter>'." + base


def extract_number(text):
    m = re.search(r"Answer:\s*\$?(-?[\d,]*\.?\d+)", text)
    if m:
        s = m.group(1)
    else:
        nums = re.findall(r"-?\d[\d,]*\.?\d*", text)
        s = nums[-1] if nums else None
    return s.replace(",", "").rstrip(".") if s else None
    
def extract_letter(text):
    m = re.search(r"Answer:\s*\(?([A-E])\b", text)
    if m:
        return m.group(1)
    m = re.search(r"\b([A-E])\b", text)
    return m.group(1) if m else None
    
def same_number(a, b):
    try:
        return float(a) == float(b)
    except (TypeError, ValueError):
        return False

def split_steps(text):
    lines = [s.strip() for s in text.split("\n\n") if s.strip()]
    return [s for s in lines if not s.startswith("Answer:")]
    
def gold_steps_count(row):
    if row["dataset"] != "gsm8k":
        return None
    idx = int(row["question_id"].split("_")[1])
    solution = gsm8k[idx]["answer"].split("####")[0]
    return len([s for s in solution.split("\n") if s.strip()])

In [6]:
def gsm_example(ex):
    solution, final = ex["answer"].split("####")
    solution = re.sub(r"<<.*?>>", "", solution).strip()
    return f"Question: {ex['question']}\n{solution}\nAnswer: {final.strip()}"

gsm_shots = "\n\n".join(gsm_example(gsm_train[i]) for i in range(K))
print(gsm_shots)

Question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
Natalia sold 48/2 = 24 clips in May.
Natalia sold 48+24 = 72 clips altogether in April and May.
Answer: 72

Question: Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?
Weng earns 12/60 = $0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $10.
Answer: 10

Question: Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet?
In the beginning, Betty has only 100 / 2 = $50.
Betty's grandparents gave her 15 * 2 = $30.
This means, Betty needs 100 - 50 - 30 - 15 = $5 more.
Answer: 5


In [7]:
for i in range(K):
    ex = arc_train[i]
    print(ex["question"], ex["choices"]["text"], ex["answerKey"], "\n")

George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat? ['dry palms', 'wet palms', 'palms covered with oil', 'palms covered with lotion'] A 

Which of the following statements best explains why magnets usually stick to a refrigerator door? ['The refrigerator door is smooth.', 'The refrigerator door contains iron.', 'The refrigerator door is a good conductor.', 'The refrigerator door has electric wires in it.'] B 

A fold observed in layers of sedimentary rock most likely resulted from the ['cooling of flowing magma.', 'converging of crustal plates.', 'deposition of river sediments.', 'solution of carbonate minerals.'] B 



In [8]:
arc_reasonings = [
    "\nRubbing creates heat through friction, and more friction produces more heat.\nDry palms have the most friction, while water, oil and lotion make the skin slippery and reduce friction.\n",
    "\nMagnets are attracted to certain metals, especially iron.\nA refrigerator door contains iron, so a magnet sticks to it, while smoothness, conductivity or electric wires do not cause magnetic attraction.\n",
    "\nFolds in sedimentary rock layers form when the layers are squeezed and bent by large forces.\nConverging crustal plates push against each other and create this pressure, while cooling magma, river deposits and dissolving minerals do not bend rock layers.\n",
]

def arc_example(ex, reasoning):
    opts = "\n".join(f"{chr(65+i)}. {o}" for i, o in enumerate(ex["choices"]["text"]))
    gold = chr(65 + ex["choices"]["label"].index(ex["answerKey"]))
    return f"Question: {ex['question']}\n{opts}\nReasoning: {reasoning}\nAnswer: {gold}"

arc_shots = "\n\n".join(arc_example(arc_train[i], arc_reasonings[i]) for i in range(K))
print(arc_shots)

Question: George wants to warm his hands quickly by rubbing them. Which skin surface will produce the most heat?
A. dry palms
B. wet palms
C. palms covered with oil
D. palms covered with lotion
Reasoning: 
Rubbing creates heat through friction, and more friction produces more heat.
Dry palms have the most friction, while water, oil and lotion make the skin slippery and reduce friction.

Answer: A

Question: Which of the following statements best explains why magnets usually stick to a refrigerator door?
A. The refrigerator door is smooth.
B. The refrigerator door contains iron.
C. The refrigerator door is a good conductor.
D. The refrigerator door has electric wires in it.
Reasoning: 
Magnets are attracted to certain metals, especially iron.
A refrigerator door contains iron, so a magnet sticks to it, while smoothness, conductivity or electric wires do not cause magnetic attraction.

Answer: B

Question: A fold observed in layers of sedimentary rock most likely resulted from the
A. coo

In [66]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", device_map="auto")
model.eval()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=896, out_features=896, bias=True)
          (k_proj): Linear(in_features=896, out_features=128, bias=True)
          (v_proj): Linear(in_features=896, out_features=128, bias=True)
          (o_proj): Linear(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e-06)
    (rotary_emb): Qwen2

In [67]:
rows = []
for i in gsm_idx:
    ex = gsm8k[i]
    gold = ex["answer"].split("####")[-1].strip().replace(",", "")
    for mode in ["direct", "cot","fewshot"]:
        p = gsm_prompt(ex["question"], mode)
        t0 = time.time()
        txt, n = generate(model,tokenizer,p, 16 if mode == "direct" else 700)
        t = time.time() - t0
        pred = extract_number(txt)
        rows.append(dict(question_id=f"gsm8k_{i}", dataset="gsm8k", model="Qwen/Qwen2.5-0.5B-Instruct", mode=mode,
                         prompt=p, generated_text=txt, extracted_answer=pred,
                         gold_answer=gold, correct=same_number(pred, gold), n_tokens=n,gen_time_s=t))

for i in arc_idx:
    ex = arc[i]
    labels = ex["choices"]["label"]
    gold = chr(65 + labels.index(ex["answerKey"]))
    for mode in ["direct", "cot","fewshot"]:
        p = arc_prompt(ex["question"], ex["choices"]["text"], mode)
        t0 = time.time()
        txt, n = generate(model,tokenizer, p, 16 if mode == "direct" else 700)
        t = time.time() - t0
        pred = extract_letter(txt)
        rows.append(dict(question_id=f"arc_{i}", dataset="arc", model="Qwen/Qwen2.5-0.5B-Instruct", mode=mode,
                         prompt=p, generated_text=txt, extracted_answer=pred,
                         gold_answer=gold, correct=(pred == gold), n_tokens=n,gen_time_s=t))

df = pd.DataFrame(rows)
df["truncated"] = (df["mode"] == "direct") & (df["n_tokens"] >= 16)|(df["mode"] != "direct") & (df["n_tokens"] >= 700)
df["steps"] = df.apply(lambda r: json.dumps(split_steps(r["generated_text"]) if r["mode"] != "direct" else [], ensure_ascii=False), axis=1)
df["n_steps"] = df["steps"].apply(lambda s: len(json.loads(s)))
df["gold_n_steps"] = df.apply(gold_steps_count, axis=1)
df.to_csv("/kaggle/working/results_test100_Qwen2.5-0.5B-Instruct.csv", index=False)
df

,question_id,dataset,model,mode,prompt,generated_text,extracted_answer,gold_answer,correct,n_tokens,gen_time_s,truncated,steps,n_steps,gold_n_steps
0,gsm8k_1309,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,direct,The girls are trying to raise money for a carn...,$1550,1550,2280,False,6,0.348003,False,[],0,3.0
1,gsm8k_1309,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,cot,The girls are trying to raise money for a carn...,To find out how much money each girl raised an...,2780,2280,False,674,28.243259,False,"[""To find out how much money each girl raised ...",19,3.0
2,gsm8k_1309,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,fewshot,Solve the problems step by step. At the end wr...,To find out how much money was raised in total...,1880,2280,False,248,10.566451,False,"[""To find out how much money was raised in tot...",9,3.0
3,gsm8k_228,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,direct,Kalinda is working on a 360 piece puzzle with ...,12,12,1,False,3,0.211403,False,[],0,4.0
4,gsm8k_228,gsm8k,Qwen/Qwen2.5-0.5B-Instruct,cot,Kalinda is working on a 360 piece puzzle with ...,To determine how long it will take Kalinda and...,60,1,False,451,18.628839,False,"[""To determine how long it will take Kalinda a...",8,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,arc_833,arc,Qwen/Qwen2.5-0.5B-Instruct,cot,Think step by step. At the end write 'Answer: ...,To determine the correct answer to the questio...,B,D,False,217,9.226007,False,"[""To determine the correct answer to the quest...",6,NaN
596,arc_833,arc,Qwen/Qwen2.5-0.5B-Instruct,fewshot,Think step by step. At the end write 'Answer: ...,"To prevent electric shock, it's crucial to fol...",C,D,False,128,5.640067,False,"[""To prevent electric shock, it's crucial to f...",5,NaN
597,arc_389,arc,Qwen/Qwen2.5-0.5B-Instruct,direct,Reply with only the letter of the correct opti...,A,A,A,True,2,0.172405,False,[],0,NaN
598,arc_389,arc,Qwen/Qwen2.5-0.5B-Instruct,cot,Think step by step. At the end write 'Answer: ...,To determine how a balanced chemical equation ...,C,A,False,342,14.508077,False,"[""To determine how a balanced chemical equatio...",8,NaN


In [68]:
df.groupby(["dataset", "mode"])[["correct", "truncated"]].mean()

correct  truncated
dataset mode                       
arc     cot         0.38       0.00
        direct      0.50       0.00
        fewshot     0.45       0.00
gsm8k   cot         0.37       0.02
        direct      0.01       0.00
        fewshot     0.46       0.00

In [70]:
df.groupby(["dataset", "mode"])["n_tokens"].mean()   

dataset  mode   
arc      cot        263.14
         direct       2.22
         fewshot    128.85
gsm8k    cot        317.37
         direct       3.68
         fewshot    237.31
Name: n_tokens, dtype: float64

In [71]:
df.groupby(["dataset", "mode"])["extracted_answer"].apply(lambda s: s.isna().mean())  

dataset  mode   
arc      cot        0.00
         direct     0.00
         fewshot    0.01
gsm8k    cot        0.00
         direct     0.00
         fewshot    0.00
Name: extracted_answer, dtype: float64

In [11]:
tokenizer_2 = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-3B-Instruct")
model_2 = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-3B-Instruct", device_map="auto")
model_2.eval()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((2048,), eps=1e-06)
    (ro

In [ ]:
rows = []
for i in gsm_idx:
    if i%5==0: print(i)
    ex = gsm8k[i]
    gold = ex["answer"].split("####")[-1].strip().replace(",", "")
    for mode in ["direct", "cot","fewshot"]:
        p = gsm_prompt(ex["question"], mode)
        t0 = time.time()
        txt, n = generate(model_2,tokenizer_2,p, 16 if mode == "direct" else 700)
        t = time.time() - t0
        pred = extract_number(txt)
        rows.append(dict(question_id=f"gsm8k_{i}", dataset="gsm8k", model="Qwen/Qwen2.5-3B-Instruct", mode=mode,
                         prompt=p, generated_text=txt, extracted_answer=pred,
                         gold_answer=gold, correct=same_number(pred, gold), n_tokens=n,gen_time_s=t))

for i in arc_idx:
    if i%5==0: print(i)
    ex = arc[i]
    labels = ex["choices"]["label"]
    gold = chr(65 + labels.index(ex["answerKey"]))
    for mode in ["direct", "cot","fewshot"]:
        p = arc_prompt(ex["question"], ex["choices"]["text"], mode)
        t0 = time.time()
        txt, n = generate(model_2,tokenizer_2, p, 16 if mode == "direct" else 700)
        t = time.time() - t0
        pred = extract_letter(txt)
        rows.append(dict(question_id=f"arc_{i}", dataset="arc", model="Qwen/Qwen2.5-3B-Instruct", mode=mode,
                         prompt=p, generated_text=txt, extracted_answer=pred,
                         gold_answer=gold, correct=(pred == gold), n_tokens=n,gen_time_s=t))

df2 = pd.DataFrame(rows)
df2["truncated"] = (df2["mode"] == "direct") & (df2["n_tokens"] >= 16)|(df2["mode"] != "direct") & (df2["n_tokens"] >= 700)
df2["steps"] = df2.apply(lambda r: json.dumps(split_steps(r["generated_text"]) if r["mode"] != "direct" else [], ensure_ascii=False), axis=1)
df2["n_steps"] = df2["steps"].apply(lambda s: len(json.loads(s)))
df2["gold_n_steps"] = df2.apply(gold_steps_count, axis=1)
df2.to_csv("/kaggle/working/results_test100_Qwen2.5-3B-Instruct.csv", index=False)
df2

In [ ]:
df2.groupby(["dataset", "mode"])[["correct", "truncated"]].mean()

In [ ]:
df2.groupby(["dataset", "mode"])["n_tokens"].mean()   

In [ ]:
df2.groupby(["dataset", "mode"])["extracted_answer"].apply(lambda s: s.isna().mean())  

In [9]:
tokenizer_3 = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
model_3 = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct", device_map="auto")
model_3.eval()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

In [11]:
rows = []
count=0
for i in gsm_idx:
    count+=1
    if count%10==0: print(count)
    ex = gsm8k[i]
    gold = ex["answer"].split("####")[-1].strip().replace(",", "")
    for mode in ["direct", "cot","fewshot"]:
        p = gsm_prompt(ex["question"], mode)
        t0 = time.time()
        txt, n = generate(model_3,tokenizer_3,p, 16 if mode == "direct" else 700)
        t = time.time() - t0
        pred = extract_number(txt)
        rows.append(dict(question_id=f"gsm8k_{i}", dataset="gsm8k", model="Qwen/Qwen2.5-7B-Instruct", mode=mode,
                         prompt=p, generated_text=txt, extracted_answer=pred,
                         gold_answer=gold, correct=same_number(pred, gold), n_tokens=n,gen_time_s=t))
count=0
for i in arc_idx:
    count+=1
    if count%10==0: print(count)
    ex = arc[i]
    labels = ex["choices"]["label"]
    gold = chr(65 + labels.index(ex["answerKey"]))
    for mode in ["direct", "cot","fewshot"]:
        p = arc_prompt(ex["question"], ex["choices"]["text"], mode)
        t0 = time.time()
        txt, n = generate(model_3,tokenizer_3, p, 16 if mode == "direct" else 700)
        t = time.time() - t0
        pred = extract_letter(txt)
        rows.append(dict(question_id=f"arc_{i}", dataset="arc", model="Qwen/Qwen2.5-7B-Instruct", mode=mode,
                         prompt=p, generated_text=txt, extracted_answer=pred,
                         gold_answer=gold, correct=(pred == gold), n_tokens=n,gen_time_s=t))

df3 = pd.DataFrame(rows)
df3["truncated"] = (df3["mode"] == "direct") & (df3["n_tokens"] >= 16)|(df3["mode"] != "direct") & (df3["n_tokens"] >= 700)
df3["steps"] = df3.apply(lambda r: json.dumps(split_steps(r["generated_text"]) if r["mode"] != "direct" else [], ensure_ascii=False), axis=1)
df3["n_steps"] = df3["steps"].apply(lambda s: len(json.loads(s)))
df3["gold_n_steps"] = df3.apply(gold_steps_count, axis=1)
df3.to_csv("/kaggle/working/results_test100_Qwen2.5-7B-Instruct.csv", index=False)
df3

10
20
40
50
60
70
80
90
100
10
20
30
40
50
60
70
80
90
100


,question_id,dataset,model,mode,prompt,generated_text,extracted_answer,gold_answer,correct,n_tokens,gen_time_s,truncated,steps,n_steps,gold_n_steps
0,gsm8k_1309,gsm8k,Qwen/Qwen2.5-7B-Instruct,direct,The girls are trying to raise money for a carn...,1880,1880,2280,False,5,1.097796,False,[],0,3.0
1,gsm8k_1309,gsm8k,Qwen/Qwen2.5-7B-Instruct,cot,The girls are trying to raise money for a carn...,Let's break down the problem step by step.\n\n...,2180,2280,False,330,22.066348,False,"[""Let's break down the problem step by step."",...",7,3.0
2,gsm8k_1309,gsm8k,Qwen/Qwen2.5-7B-Instruct,fewshot,Solve the problems step by step. At the end wr...,"First, let's determine how much each person ra...",2180,2280,False,142,12.018824,False,"[""First, let's determine how much each person ...",4,3.0
3,gsm8k_228,gsm8k,Qwen/Qwen2.5-7B-Instruct,direct,Kalinda is working on a 360 piece puzzle with ...,2.5,2.5,1,False,4,0.883763,False,[],0,4.0
4,gsm8k_228,gsm8k,Qwen/Qwen2.5-7B-Instruct,cot,Kalinda is working on a 360 piece puzzle with ...,Let's break down the problem step by step.\n\n...,1,1,True,256,17.202082,False,"[""Let's break down the problem step by step."",...",7,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
595,arc_833,arc,Qwen/Qwen2.5-7B-Instruct,cot,Think step by step. At the end write 'Answer: ...,"To determine the correct answer, let's analyze...",D,D,True,191,12.943899,False,"[""To determine the correct answer, let's analy...",6,NaN
596,arc_833,arc,Qwen/Qwen2.5-7B-Instruct,fewshot,Think step by step. At the end write 'Answer: ...,Reading all directions first ensures that you ...,D,D,True,62,6.812202,False,"[""Reading all directions first ensures that yo...",1,NaN
597,arc_389,arc,Qwen/Qwen2.5-7B-Instruct,direct,Reply with only the letter of the correct opti...,A,A,A,True,2,0.825088,False,[],0,NaN
598,arc_389,arc,Qwen/Qwen2.5-7B-Instruct,cot,Think step by step. At the end write 'Answer: ...,"To answer this question, let's break it down s...",A,A,True,271,18.411574,False,"[""To answer this question, let's break it down...",4,NaN


In [15]:
df3.groupby(["dataset", "mode"])[["correct", "truncated"]].mean()

correct  truncated
dataset mode                       
arc     cot         0.95        0.0
        direct      0.93        0.0
        fewshot     0.93        0.0
gsm8k   cot         0.94        0.0
        direct      0.19        0.0
        fewshot     0.92        0.0

In [16]:
df3.groupby(["dataset", "mode"])["n_tokens"].mean()   

dataset  mode   
arc      cot        263.00
         direct       2.00
         fewshot     83.82
gsm8k    cot        263.17
         direct       3.47
         fewshot    186.06
Name: n_tokens, dtype: float64

In [17]:
df3.groupby(["dataset", "mode"])["extracted_answer"].apply(lambda s: s.isna().mean())  

dataset  mode   
arc      cot        0.0
         direct     0.0
         fewshot    0.0
gsm8k    cot        0.0
         direct     0.0
         fewshot    0.0
Name: extracted_answer, dtype: float64